# Kestrel V3, step 3: MTP ablation

**Pre-registered rule (docs/11 §1):** two matched runs from the same checkpoint, with the same data,
schedule and optimizer start, and only `--mtp` differing. **Keep MTP only if it beats the control on
`val_mean` at matched compute**, counting its extra time per step.

A100, `Run all`. About 3 hours and ~20 units. Each arm resumes from Drive after a disconnect.


In [ ]:
# 0. GPU check, then the code checks. A failed test stops here, before the corpus is copied.
import os, json, shutil, glob, subprocess, time
import torch
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU'
print(torch.__version__, '|', name)
if 'A100' not in name and 'H100' not in name:
    raise SystemExit(f'Got {name}, not an A100. Runtime -> Change runtime type -> A100, then rerun.')
REPO = '/content/Kestrel'
if not os.path.isdir(REPO):
    !git clone -q https://github.com/YoMosa2009/Kestrel.git {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}
!git log --oneline -1
!pip -q install tokenizers
r = subprocess.run(['python', '-m', 'scripts.test_mtp'], capture_output=True, text=True, env={**os.environ, 'CUDA_VISIBLE_DEVICES': ''})
print(r.stdout[-600:], r.stderr[-1500:])
assert 'MTP_TEST_OK' in r.stdout, 'MTP tests failed - see above'


In [ ]:
# 1. Drive, and the corpus onto local disk
from google.colab import drive
try:
    drive.mount('/content/drive')
except ValueError as e:
    print('first mount failed, retrying:', e)
    drive.mount('/content/drive', force_remount=True)
KD = '/content/drive/MyDrive/Kestrel'
assert os.path.isdir(KD), f'{KD} not found - mounted the wrong Google account?'
CTX = f'{KD}/runs/ctx4k/ckpt.pt'
assert os.path.exists(CTX) and os.path.exists(f'{KD}/datasets/corpus-5b/FILES.json')
t = time.time()
os.makedirs('/content/data_5b', exist_ok=True)
for f in sorted(glob.glob(f'{KD}/datasets/corpus-5b/*')):
    out = f'/content/data_5b/{os.path.basename(f)}'
    if not os.path.exists(out) or os.path.getsize(out) != os.path.getsize(f):
        shutil.copy(f, out)
print(f'corpus: {sum(os.path.getsize(p) for p in glob.glob("/content/data_5b/*"))/1e9:.2f} GB, {time.time()-t:.0f}s')


In [ ]:
# 2. The two arms, control first. Same flags apart from --mtp. 350 steps x 196,608 tokens = 69M
#    tokens each, at the LR planned for step 4, from the pre-SFT 4k checkpoint. The curriculum is
#    held near the anneal mix, the data distribution that checkpoint finished on.
CTX_STEP = 19590
STEPS = 350
def arm(name, extra):
    out = f'{KD}/runs/mtp_{name}'
    os.makedirs(out, exist_ok=True)
    st = f'{out}/status.json'
    if os.path.exists(st) and json.load(open(st)).get('state') == 'done':
        print(name, 'already done'); return
    cmd = (f'python -u -m kestrel.train --preset nano --data-dir /content/data_5b --curriculum --curriculum-start 0.85 '
           f'--bf16 --grad-checkpoint --fused-adam --prefetch 3 --seq 4096 --batch 16 --accum 3 '
           f'--steps {CTX_STEP + STEPS} --warmup 30 --decay-frac 0.3 --muon-lr 0.005 --adamw-lr 0.00075 '
           f'--rope-scale 4 --init-from {CTX} --out {out} --log-every 10 --eval-every 70 --eval-iters 6 '
           f'--ckpt-every 100 --ckpt-secs 1800 {extra}')
    get_ipython().system(cmd)
    assert json.load(open(st)).get('state') == 'done', f'{name} did not finish. Run all again to resume.'
arm('control', '')
print('CONTROL_DONE')
arm('mtp', '--mtp')
print('MTP_DONE')


In [ ]:
# 3. Verdict inputs: val_mean on the same suite at seq 4096, the per-domain curves, and speed.
GEN = f'{KD}/evals/mtp_generalization.json'
have = json.load(open(GEN)) if os.path.exists(GEN) else {}
def rows(name):
    return [json.loads(l) for l in open(f'{KD}/runs/mtp_{name}/metrics.jsonl')]
for name in ('control', 'mtp'):
    if not (isinstance(have, dict) and name in json.dumps(have)):
        !python -m scripts.eval_generalization --ckpt {KD}/runs/mtp_{name}/ckpt.pt --data-dir /content/data_5b --seq 4096 --label {name} --out {GEN}
print(open(GEN).read()[-1800:])
for name in ('control', 'mtp'):
    r = rows(name)
    speed = [x['tok_s'] for x in r if 'tok_s' in x]
    print('SPEED', name, round(sum(speed[1:]) / max(1, len(speed) - 1)))
    for x in r:
        if 'val_by_domain' in json.dumps(x) or 'probes' in x:
            print('CURVE', name, json.dumps(x)[:700])
print('STEP3_DONE')
